# **Implement an LLM-based code generation system and evaluate its ability to generate executable code.**

# Task
Implement an LLM-based code generation system and evaluate its ability to generate executable code.

In [ ]:
%pip install torch transformers accelerate -q

In [ ]:
import torch
import time
import subprocess
import tempfile
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# --------------------------------------------------
# 1. Load lightweight open-source Code LLM
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# --------------------------------------------------
# 2. Programming tasks
# --------------------------------------------------

tasks = [
    {
        "prompt": "Write a Python function factorial(n) that returns n factorial.",
        "test": "assert factorial(5) == 120"
    },
    {
        "prompt": "Write a Python function is_prime(n) that returns True if n is prime.",
        "test": "assert is_prime(7) == True"
    },
    {
        "prompt": "Write a Python function reverse_string(s) that reverses a string.",
        "test": "assert reverse_string('hello') == 'olleh'"
    },
    {
        "prompt": "Write a Python function square(n) that returns the square of n.",
        "test": "assert square(6) == 36"
    },
    {
        "prompt": "Write a Python function add(a,b) that returns the sum of a and b.",
        "test": "assert add(10,20) == 30"
    }
]

# --------------------------------------------------
# 3. Generate code
# --------------------------------------------------

def generate_code(prompt):

    messages = [
        {
            "role": "system",
            "content": "You are an expert Python programmer. Return only Python code."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    start = time.time()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    latency = time.time() - start

    code = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # Remove markdown code blocks
    code = code.replace("```python", "").replace("```", "").strip()

    return code, latency


# --------------------------------------------------
# 4. Execute generated code
# --------------------------------------------------

def evaluate(code, test):

    program = code + "\n" + test

    try:
        with tempfile.NamedTemporaryFile(
            suffix=".py",
            mode="w",
            delete=False
        ) as f:

            f.write(program)
            filename = f.name

        result = subprocess.run(
            ["python", filename],
            capture_output=True,
            timeout=5
        )

        return result.returncode == 0

    except Exception:
        return False


# --------------------------------------------------
# 5. Run experiment
# --------------------------------------------------

correct = 0
total_time = 0

for i, task in enumerate(tasks, 1):

    code, latency = generate_code(task["prompt"])

    success = evaluate(
        code,
        task["test"]
    )

    correct += success
    total_time += latency

    print("\n" + "=" * 50)
    print("TASK:", i)
    print("Generated Code:")
    print(code)
    print("Result:", "PASS" if success else "FAIL")
    print("Latency:", round(latency, 3), "seconds")


# --------------------------------------------------
# 6. Final results
# --------------------------------------------------

accuracy = correct / len(tasks) * 100
avg_latency = total_time / len(tasks)

print("\n" + "=" * 50)
print("FINAL RESULTS")
print("=" * 50)

print("Tasks:", len(tasks))
print("Correct:", correct)
print("Functional Accuracy:", round(accuracy, 2), "%")
print("Average Latency:", round(avg_latency, 3), "seconds")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


TASK: 1
Generated Code:
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
Result: PASS
Latency: 2.607 seconds

TASK: 2
Generated Code:
def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
Result: PASS
Latency: 9.253 seconds

TASK: 3
Generated Code:
def reverse_string(s):
    return s[::-1]
Result: PASS
Latency: 0.637 seconds

TASK: 4
Generated Code:
def square(n):
    return n * n
Result: PASS
Latency: 0.593 seconds

TASK: 5
Generated Code:
def add(a, b):
    return a + b
Result: PASS
Latency: 0.691 seconds

FINAL RESULTS
Tasks: 5
Correct: 5
Functional Accuracy: 100.0 %
Average Latency: 2.756 seconds
